# Assignment 8

### Embedding Models
Small: sentence-transformers/all-MiniLM-L6-v2
- Maximum Sequence Length: 256 Tokens

Medium: sentence-transformers/all-mpnet-base-v2
- Maximum Sequence Length: 384 Tokens

Large: BAAI/bge-large-en-v1.5
- Maximum Sequence Length: 512 Tokens


### Generation Model
Qwen/Qwen2.5-3B-Instruct

In [20]:
from sentence_transformers import SentenceTransformer
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

In [21]:
model_names = [
    "sentence-transformers/all-MiniLM-L6-v2", # Small
    "sentence-transformers/all-mpnet-base-v2", # Medium
    "BAAI/bge-large-en-v1.5"                   # Large
]

In [22]:
small_model = SentenceTransformer(model_names[0])
medium_model = SentenceTransformer(model_names[1])
large_model = SentenceTransformer(model_names[2])

embedding_models = {
    "small": small_model,
    "medium": medium_model,
    "large": large_model
}

print("Loaded models:")
for name, model in embedding_models.items():
    print(f"{name}: {model}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5850.51it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7058.25it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 8459.75it/s]
BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_

Loaded models:
small: SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False, 'architecture': 'BertModel'})
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
)
medium: SentenceTransformer(
  (0): Transformer({'max_seq_length': 384, 'do_lower_case': False, 'architecture': 'MPNetModel'})
  (1): Pooling({'word_embedding_dimension': 768, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
)
large: SentenceTransformer(
  (0): Transformer({'max_seq_length': 512, 'do_lower_case': True, 'archit

Text Representation

In [23]:
df = pd.read_csv("uiuc_student_marketplace_knowledge_base.csv")
df.head()

,paragraph_id,listing_type,title,paragraph_text
0,1,Service,Calculus and Linear Algebra Tutoring,A UIUC junior in engineering offers personaliz...
1,2,Service,Python and Java Programming Tutoring,A UIUC computer science student provides one-o...
2,3,Service,Chemistry and Biology Exam Prep,A pre-med student offers tutoring for general ...
3,4,Service,Graduation Photography Sessions,A student photographer offers graduation photo...
4,5,Service,Event and Club Photography,A UIUC student photographer provides event cov...


In [24]:
df["word_count"] = df["paragraph_text"].apply(lambda x: len(str(x).split()))

In [25]:
paragraphs = df.to_dict(orient="records")

Chunking Strategies

In [26]:
# Fixed Length Chunking
def fixed_length_chunking(paragraphs, target_words=300, max_words=360):
    chunks = []
    current_chunk = []
    current_word_count = 0
    chunk_num = 1

    for para in paragraphs:
        para_text = para["paragraph_text"]
        para_words = para["word_count"]

        # If current chunk is empty, always start with this paragraph
        if not current_chunk:
            current_chunk.append(para)
            current_word_count += para_words
            continue

        # If adding this paragraph keeps us within the soft target/max, add it
        if current_word_count + para_words <= max_words:
            current_chunk.append(para)
            current_word_count += para_words
        else:
            # Save the current chunk
            chunk_text = " ".join([p["paragraph_text"] for p in current_chunk])

            chunks.append({
                "chunk_id": f"fixed_{chunk_num:03d}",
                "strategy": "fixed",
                "paragraph_ids": [p["paragraph_id"] for p in current_chunk],
                "text": chunk_text,
                "word_count": current_word_count
            })

            # Start a new chunk with the current paragraph
            chunk_num += 1
            current_chunk = [para]
            current_word_count = para_words

    # Save final chunk
    if current_chunk:
        chunk_text = " ".join([p["paragraph_text"] for p in current_chunk])
        chunks.append({
            "chunk_id": f"fixed_{chunk_num:03d}",
            "strategy": "fixed",
            "paragraph_ids": [p["paragraph_id"] for p in current_chunk],
            "text": chunk_text,
            "word_count": current_word_count
        })

    return chunks

In [27]:
fixed_chunks = fixed_length_chunking(paragraphs, target_words=300, max_words=360)

print("Number of fixed chunks:", len(fixed_chunks))
print()
print(fixed_chunks[0]["chunk_id"])
print(fixed_chunks[0]["paragraph_ids"])
print(fixed_chunks[0]["word_count"])
print(fixed_chunks[0]["text"])

Number of fixed chunks: 14

fixed_001
[1, 2, 3]
339
A UIUC junior in engineering offers personalized tutoring in calculus and linear algebra for students taking introductory and intermediate math courses. Sessions are designed around the student's syllabus, homework set, and exam schedule, with extra practice on topics such as derivatives, integration techniques, vector spaces, matrices, determinants, and eigenvalues. The tutor focuses on explaining why methods work instead of only showing formulas, so students can solve unfamiliar problems independently. Meetings can be held at Grainger, the Main Library, or over Zoom, and review packets can be created before midterms. This service is especially useful for students who want structured weekly support, last-minute exam preparation, or patient help working through difficult lecture material. A UIUC computer science student provides one-on-one programming tutoring for beginners learning Python or Java. The service helps students understan

In [28]:
# Overlapping Paragraph Chunking

def overlapping_paragraph_chunking(paragraphs, window_size=2, stride=1):
    chunks = []
    chunk_num = 1

    for i in range(0, len(paragraphs) - window_size + 1, stride):
        window = paragraphs[i:i + window_size]
        chunk_text = " ".join(p["paragraph_text"] for p in window)

        chunks.append({
            "chunk_id": f"overlap_{chunk_num:03d}",
            "strategy": "overlap",
            "paragraph_ids": [p["paragraph_id"] for p in window],
            "text": chunk_text,
            "word_count": sum(p["word_count"] for p in window)
        })

        chunk_num += 1

    return chunks

In [29]:
overlap_chunks = overlapping_paragraph_chunking(paragraphs, window_size=2, stride=1)

print("Number of overlap chunks:", len(overlap_chunks))
print(overlap_chunks[0]["chunk_id"])
print(overlap_chunks[0]["paragraph_ids"])
print(overlap_chunks[0]["word_count"])
print(overlap_chunks[0]["text"][:500])

Number of overlap chunks: 39
overlap_001
[1, 2]
225
A UIUC junior in engineering offers personalized tutoring in calculus and linear algebra for students taking introductory and intermediate math courses. Sessions are designed around the student's syllabus, homework set, and exam schedule, with extra practice on topics such as derivatives, integration techniques, vector spaces, matrices, determinants, and eigenvalues. The tutor focuses on explaining why methods work instead of only showing formulas, so students can solve unfamiliar problems indep


In [30]:
# Hybrid / Strategy Chunking

def assign_semantic_category(row):
    title = row["title"].lower()
    listing_type = row["listing_type"].lower()

    if listing_type == "service":
        if any(word in title for word in ["tutor", "programming", "language", "music", "resume"]):
            return "education"
        elif any(word in title for word in ["photo", "video", "design", "website"]):
            return "creative"
        elif any(word in title for word in ["hair", "nail", "makeup"]):
            return "beauty"
        elif any(word in title for word in ["pet", "dog"]):
            return "pet_care"
        elif any(word in title for word in ["moving", "cleaning", "laundry", "bike", "tech"]):
            return "student_help"
        else:
            return "other_services"

    elif listing_type == "product":
        if any(word in title for word in ["laptop", "monitor", "camera", "printer", "headphones", "console"]):
            return "electronics"
        elif any(word in title for word in ["chair", "bed", "mattress", "sofa", "table", "lamp"]):
            return "furniture_home"
        elif any(word in title for word in ["backpack", "sneakers", "jacket"]):
            return "fashion"
        elif any(word in title for word in ["fridge", "microwave", "kitchen"]):
            return "appliances_kitchen"
        else:
            return "other_products"

    return "uncategorized"

In [31]:
df["semantic_category"] = df.apply(assign_semantic_category, axis=1)
paragraphs = df.to_dict(orient="records")

In [32]:
def hybrid_chunking(paragraphs, max_words=350):
    chunks = []
    chunk_num = 1

    grouped = {}
    for para in paragraphs:
        category = para["semantic_category"]
        grouped.setdefault(category, []).append(para)

    for category, items in grouped.items():
        current_chunk = []
        current_word_count = 0

        for para in items:
            para_words = para["word_count"]

            if not current_chunk:
                current_chunk.append(para)
                current_word_count += para_words
                continue

            if current_word_count + para_words <= max_words:
                current_chunk.append(para)
                current_word_count += para_words
            else:
                chunk_text = " ".join(p["paragraph_text"] for p in current_chunk)

                chunks.append({
                    "chunk_id": f"hybrid_{chunk_num:03d}",
                    "strategy": "hybrid",
                    "semantic_category": category,
                    "paragraph_ids": [p["paragraph_id"] for p in current_chunk],
                    "text": chunk_text,
                    "word_count": current_word_count
                })

                chunk_num += 1
                current_chunk = [para]
                current_word_count = para_words

        if current_chunk:
            chunk_text = " ".join(p["paragraph_text"] for p in current_chunk)

            chunks.append({
                "chunk_id": f"hybrid_{chunk_num:03d}",
                "strategy": "hybrid",
                "semantic_category": category,
                "paragraph_ids": [p["paragraph_id"] for p in current_chunk],
                "text": chunk_text,
                "word_count": current_word_count
            })

            chunk_num += 1

    return chunks

In [33]:
hybrid_chunks = hybrid_chunking(paragraphs, max_words=350)

print("Number of hybrid chunks:", len(hybrid_chunks))
print(hybrid_chunks[0]["chunk_id"])
print(hybrid_chunks[0]["semantic_category"])
print(hybrid_chunks[0]["paragraph_ids"])
print(hybrid_chunks[0]["word_count"])
print(hybrid_chunks[0]["text"][:500])

Number of hybrid chunks: 19
hybrid_001
education
[1, 2, 11]
341
A UIUC junior in engineering offers personalized tutoring in calculus and linear algebra for students taking introductory and intermediate math courses. Sessions are designed around the student's syllabus, homework set, and exam schedule, with extra practice on topics such as derivatives, integration techniques, vector spaces, matrices, determinants, and eigenvalues. The tutor focuses on explaining why methods work instead of only showing formulas, so students can solve unfamiliar problems indep


In [34]:
print("Fixed chunks:", len(fixed_chunks))
print("Overlap chunks:", len(overlap_chunks))
print("Hybrid chunks:", len(hybrid_chunks))

Fixed chunks: 14
Overlap chunks: 39
Hybrid chunks: 19


In [35]:
def preview_chunks(chunks, n=2):
    for chunk in chunks[:n]:
        print("=" * 80)
        print("Chunk ID:", chunk["chunk_id"])
        print("Paragraph IDs:", chunk["paragraph_ids"])
        if "semantic_category" in chunk:
            print("Category:", chunk["semantic_category"])
        print("Word Count:", chunk["word_count"])
        print(chunk["text"][:600], "...\n")

print("FIXED CHUNKS")
preview_chunks(fixed_chunks)

print("OVERLAP CHUNKS")
preview_chunks(overlap_chunks)

print("HYBRID CHUNKS")
preview_chunks(hybrid_chunks)

FIXED CHUNKS
Chunk ID: fixed_001
Paragraph IDs: [1, 2, 3]
Word Count: 339
A UIUC junior in engineering offers personalized tutoring in calculus and linear algebra for students taking introductory and intermediate math courses. Sessions are designed around the student's syllabus, homework set, and exam schedule, with extra practice on topics such as derivatives, integration techniques, vector spaces, matrices, determinants, and eigenvalues. The tutor focuses on explaining why methods work instead of only showing formulas, so students can solve unfamiliar problems independently. Meetings can be held at Grainger, the Main Library, or over Zoom, and review packets can b ...

Chunk ID: fixed_002
Paragraph IDs: [4, 5, 6]
Word Count: 349
A student photographer offers graduation photo sessions around popular UIUC landmarks including the Quad, Alma Mater, Foellinger Auditorium, and the Main Library steps. Each session includes pose guidance, candid shots, solo portraits, and small group photos 

In [36]:
fixed_chunks_df = pd.DataFrame(fixed_chunks)
overlap_chunks_df = pd.DataFrame(overlap_chunks)
hybrid_chunks_df = pd.DataFrame(hybrid_chunks)

fixed_chunks_df.to_csv("fixed_chunks.csv", index=False)
overlap_chunks_df.to_csv("overlap_chunks.csv", index=False)
hybrid_chunks_df.to_csv("hybrid_chunks.csv", index=False)

Embedding Pipeline

In [37]:
chunk_sets = {
    "fixed": fixed_chunks,
    "overlap": overlap_chunks,
    "hybrid": hybrid_chunks
}

embedded_results = {}

for chunking_name, chunks in chunk_sets.items():
    embedded_results[chunking_name] = {}
    texts = [chunk["text"] for chunk in chunks]

    for model_size, model in embedding_models.items():
        embeddings = model.encode(
            texts,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=True
        )

        embedded_results[chunking_name][model_size] = {
            "chunks": chunks,
            "texts": texts,
            "embeddings": embeddings
        }

        print(f"Done: {chunking_name} + {model_size} -> {embeddings.shape}")

vector_stores = {}

for chunking_name, model_data in embedded_results.items():
    vector_stores[chunking_name] = {}

    for model_size, data in model_data.items():
        vector_stores[chunking_name][model_size] = {
            "chunks": data["chunks"],
            "texts": data["texts"],
            "embeddings": data["embeddings"]
        }

        print(
            f"Stored: {chunking_name} + {model_size} -> "
            f"{data['embeddings'].shape[0]} chunks, dim={data['embeddings'].shape[1]}"
        )

Batches: 100%|██████████| 1/1 [00:03<00:00,  3.68s/it]


Done: fixed + small -> (14, 384)


Batches: 100%|██████████| 1/1 [00:04<00:00,  4.71s/it]


Done: fixed + medium -> (14, 768)


Batches: 100%|██████████| 1/1 [00:12<00:00, 12.87s/it]


...

Done: hybrid + small -> (19, 384)


Batches: 100%|██████████| 1/1 [01:04<00:00, 64.36s/it]


Done: hybrid + medium -> (19, 768)


Batches: 100%|██████████| 1/1 [06:18<00:00, 378.33s/it]

Done: hybrid + large -> (19, 1024)
Stored: fixed + small -> 14 chunks, dim=384
Stored: fixed + medium -> 14 chunks, dim=768
Stored: fixed + large -> 14 chunks, dim=1024
Stored: overlap + small -> 39 chunks, dim=384
Stored: overlap + medium -> 39 chunks, dim=768
Stored: overlap + large -> 39 chunks, dim=1024
Stored: hybrid + small -> 19 chunks, dim=384
Stored: hybrid + medium -> 19 chunks, dim=768
Stored: hybrid + large -> 19 chunks, dim=1024


Retrieval System

In [38]:
def retrieve_top_k(query, chunking_name="fixed", model_size="small", k=3):
    """
    query: user question string
    chunking_name: 'fixed', 'overlap', or 'hybrid'
    model_size: 'small', 'medium', or 'large'
    k: number of chunks to retrieve
    """
    # get the correct embedding model and stored embeddings
    model = embedding_models[model_size]
    store = vector_stores[chunking_name][model_size]

    # Query -> embedding
    query_embedding = model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    # similarity search
    similarities = cosine_similarity(query_embedding, store["embeddings"])[0]

    # top-k retrieval
    top_indices = np.argsort(similarities)[::-1][:k]

    results = []
    for rank, idx in enumerate(top_indices, start=1):
        chunk = store["chunks"][idx]

        result = {
            "rank": rank,
            "score": float(similarities[idx]),
            "chunk_id": chunk["chunk_id"],
            "paragraph_ids": chunk["paragraph_ids"],
            "text": chunk["text"]
        }

        if "semantic_category" in chunk:
            result["semantic_category"] = chunk["semantic_category"]

        results.append(result)

    return results

In [39]:
def print_retrieval_results(results, max_chars=700):
    for r in results:
        print("=" * 100)
        print(f"Rank: {r['rank']}")
        print(f"Score: {r['score']:.4f}")
        print(f"Chunk ID: {r['chunk_id']}")
        print(f"Paragraph IDs: {r['paragraph_ids']}")
        if "semantic_category" in r:
            print(f"Semantic Category: {r['semantic_category']}")
        print("Retrieved Text:")
        print(r["text"][:max_chars])
        print()

In [40]:
# test
test_queries = [
    "I need tutoring for calculus and linear algebra",
    "Who offers graduation photography on campus?",
    "I need someone to walk my dog during the week",
    "I am looking for a used laptop for coursework",
    "Do any listings offer hair styling or makeup for events?"
]

for q in test_queries:
    print("\n" + "#" * 120)
    print("QUERY:", q)
    results = retrieve_top_k(q, chunking_name="fixed", model_size="medium", k=3)
    print_retrieval_results(results, max_chars=500)


########################################################################################################################
QUERY: I need tutoring for calculus and linear algebra
Rank: 1
Score: 0.5315
Chunk ID: fixed_001
Paragraph IDs: [1, 2, 3]
Retrieved Text:
A UIUC junior in engineering offers personalized tutoring in calculus and linear algebra for students taking introductory and intermediate math courses. Sessions are designed around the student's syllabus, homework set, and exam schedule, with extra practice on topics such as derivatives, integration techniques, vector spaces, matrices, determinants, and eigenvalues. The tutor focuses on explaining why methods work instead of only showing formulas, so students can solve unfamiliar problems indep

Rank: 2
Score: 0.3974
Chunk ID: fixed_007
Paragraph IDs: [19, 20, 21]
Retrieved Text:
A UIUC student provides tech support for common laptop, tablet, and phone issues affecting school or personal use. Services include software installatio

Generation Pipeline

In [42]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

generation_model_name = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(generation_model_name)

generation_model = AutoModelForCausalLM.from_pretrained(
    generation_model_name,
    torch_dtype=torch.float32
)

device = "cuda" if torch.cuda.is_available() else "cpu"
generation_model = generation_model.to(device)

print("Loaded generation model:", generation_model_name)
print("Using device:", device)

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 434/434 [00:27<00:00, 15.55it/s]


Loaded generation model: Qwen/Qwen2.5-3B-Instruct
Using device: cpu


In [43]:
def build_rag_prompt(query, retrieved_results, max_chars_per_chunk=250):
    retrieved_context = "\n\n".join(
        [f"[Chunk {i+1}] {r['text'][:max_chars_per_chunk]}" for i, r in enumerate(retrieved_results)]
    )

    system_instruction = (
        "You are a helpful assistant for a UIUC student marketplace. "
        "Answer the user's question using only the retrieved context. "
        "Do not make up information. If the answer is not in the context, say so clearly."
    )

    prompt = f"""System Instruction:
{system_instruction}

Retrieved Context:
{retrieved_context}

User Query:
{query}

Answer:
"""
    return prompt

In [45]:
def generate_rag_answer(query, chunking_name="fixed", model_size="small", k=2, max_new_tokens=60):
    retrieved_results = retrieve_top_k(
        query=query,
        chunking_name=chunking_name,
        model_size=model_size,
        k=k
    )

    prompt = build_rag_prompt(query, retrieved_results)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )
    inputs = {key: value.to(device) for key, value in inputs.items()}

    with torch.no_grad():
        outputs = generation_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_text = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    return {
        "query": query,
        "chunking_name": chunking_name,
        "embedding_model_size": model_size,
        "retrieved_results": retrieved_results,
        "prompt": prompt,
        "final_answer": generated_text.strip()
    }

In [46]:
result = generate_rag_answer(
    query="Who offers graduation photography on campus?",
    chunking_name="fixed",
    model_size="small",
    k=1,
    max_new_tokens=30
)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


KeyboardInterrupt: 